# Failure routing: Kaggle setup and baselines

Before running: in the right sidebar under **Session options**, set **Accelerator = GPU T4 x2** and **Internet = On**. Run the cells in order.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code from GitHub

For a private repo, add your GitHub token under **Add-ons → Secrets** with the name `GITHUB_TOKEN` first. Replace `YOUR_GITHUB_USERNAME` below.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, subprocess

GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <- change this
REPO = "failure-routing"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"
if not os.path.exists(f"/kaggle/working/{REPO}"):
    subprocess.run(["git", "clone", url, f"/kaggle/working/{REPO}"], check=True)
else:
    subprocess.run(["git", "-C", f"/kaggle/working/{REPO}", "pull"], check=True)
os.chdir(f"/kaggle/working/{REPO}")
print(os.getcwd())

## 3. Install packages

This takes several minutes. Run it before importing `torch` anywhere in this session. If vLLM fails to install or crash later, skip it and use `--backend hf` in step 6 (slower).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q vllm

## 4. Run the tests (checker and metrics)

In [ ]:
!python -m pytest tests/ -q

## 5. Check the data loads

In [ ]:
!python src/data.py

## 6. Smoke test: 20 problems, 4 samples each

This also tells you how long the full run will take: multiply the generation time by about 100.

In [ ]:
!python scripts/eval_baseline.py --shots 0 --limit 20 --n_samples 4 --name smoke
# If vLLM fails, use the slower fallback instead:
# !python scripts/eval_baseline.py --shots 0 --limit 20 --n_samples 4 --name smoke --backend hf

## 7. Full baselines on MATH500 (16 samples per problem)

For long runs, use **Save Version → Save & Run All (Commit)** so the notebook keeps running after you close the tab. Results are saved under `results/`.

In [ ]:
!python scripts/eval_baseline.py --shots 0 --name base_0shot

In [ ]:
!python scripts/eval_baseline.py --shots 4 --name base_4shot

## 8. Look at the results

In [ ]:
import json
for name in ["base_0shot", "base_4shot"]:
    path = f"results/{name}/summary.json"
    if os.path.exists(path):
        s = json.load(open(path))
        print(name, "pass@k:", s["pass@k"], "| confident errors (>=12/16):",
              s["confident_error_rate"].get("12"), "| format validity:", s["format_validity"])